# 02 · Main task: predict the nightly price (log EUR)

**Question.** Does merging external open data improve the prediction of a listing's quoted nightly price?

| Set | Features |
|---|---|
| A | listing attributes only (16) |
| A + one source | A plus one external source |
| **B** | A + all five external sources, **no coordinates** (78) |
| B − one source | leave-one-source-out (does a source carry unique information?) |
| C | A + latitude, longitude, arrondissement (reference for "all location information") |
| D | C + all external |

**Merging gain = B vs A.** B does not get coordinates on purpose: coordinates already sit in the Airbnb table, so they are not external data, and once they are in the model a station distance adds nothing (that is why the earlier test, D vs C, found zero).

**Evaluation.** 5-fold grouped CV with three groupings: unseen hosts; unseen 1 km cells; unseen 3 km blocks. For the two spatial ones, hosts of the test fold are also removed from training. Same gradient boosting settings for every set (`src/common.py`).

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np, pandas as pd
from config import CLASSMATE_REPO, SLIM, EXT, OUT
from common import *
from external import build
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
st = pd.read_csv(CLASSMATE_REPO / "data" / "stations_used.csv")   # official IDFM stations (classmate's repo)
print("classmate repo:", CLASSMATE_REPO.exists(), "| slim:", SLIM.exists(), "| external:", EXT.exists())

classmate repo: True | slim: True | external: True


In [2]:
df = pd.read_csv(CLASSMATE_REPO / "data" / "price_features.csv.gz")
X, G, info, _ = build(df.latitude.values, df.longitude.values, EXT, st)
df = pd.concat([df, X], axis=1)
df["cell"] = cell_1km(df.latitude, df.longitude); df["block"] = block_3km(df.latitude, df.longitude)
cat(df, ["property_type", "neighbourhood_cleansed"])
print(len(df), "listings |", df.cell.nunique(), "1 km cells |", df.block.nunique(), "3 km blocks")

31044 listings | 120 1 km cells | 21 3 km blocks


In [3]:
A = ["accommodates", "bedrooms", "beds", "bathrooms_n", "minimum_nights", "amenity_count", "wifi", "kitchen", "washer",
     "air_conditioning", "elevator", "quote_stay_nights", "quote_month_sin", "quote_month_cos", "quote_lead_days", "property_type"]
EXT_ALL = sum(G.values(), []); GEO = ["latitude", "longitude", "neighbourhood_cleansed"]
S = {"A internal": A, **{f"A+{k}": A + v for k, v in G.items()}, "B A+all external": A + EXT_ALL,
     **{f"B-{k}": A + [c for c in EXT_ALL if c not in G[k]] for k in G}, "C A+coordinates": A + GEO, "D C+all external": A + GEO + EXT_ALL}
CORE = ["A internal", "B A+all external", "C A+coordinates", "D C+all external"]
{k: len(v) for k, v in S.items()}

{'A internal': 16,
 'A+stations': 26,
 'A+osm': 45,
 'A+dvf': 23,
 'A+iris_income': 26,
 'A+crime': 22,
 'B A+all external': 78,
 'B-stations': 68,
 'B-osm': 49,
 'B-dvf': 71,
 'B-iris_income': 68,
 'B-crime': 72,
 'C A+coordinates': 19,
 'D C+all external': 81}

## 1. Cross-validated R² (about 5 minutes)

In [4]:
y = np.log(df.price_eur.values); rows = []
P = run_task("T1 log price", df, y, "reg", S, rows, protocols=("host", "spatial", "block"), only={"block": CORE})
t1 = pd.DataFrame(rows); t1.to_csv(OUT / "t1_price.csv", index=False)
t1.pivot_table(index="features", columns="protocol", values="R2", sort=False)[["host", "spatial", "block"]].round(3)

T1 log price                 host     A internal                         R2=0.5434  MAE=0.2954


T1 log price                 host     A+stations                         R2=0.6131  MAE=0.2686


T1 log price                 host     A+osm                              R2=0.6495  MAE=0.2541


T1 log price                 host     A+dvf                              R2=0.6483  MAE=0.2539


T1 log price                 host     A+iris_income                      R2=0.6368  MAE=0.2587


T1 log price                 host     A+crime                            R2=0.6328  MAE=0.2604


... (23 outputs left out)

T1 log price                 block    B A+all external                   R2=0.6388  MAE=0.2581


T1 log price                 block    C A+coordinates                    R2=0.6096  MAE=0.2691


T1 log price                 block    D C+all external                   R2=0.6362  MAE=0.2588


protocol,host,spatial,block
features,,,
A internal,0.543,0.535,0.522
A+stations,0.613,0.587,NaN
A+osm,0.650,0.632,NaN
A+dvf,0.648,0.640,NaN
A+iris_income,0.637,0.616,NaN
A+crime,0.633,0.619,NaN
B A+all external,0.654,0.646,0.639
B-stations,0.654,0.646,NaN
B-osm,0.654,0.645,NaN


**Stability across folds** (gain in R² per fold; all five folds should be positive for a robust claim).

In [5]:
def fold_gain(a, b, proto):
    s = t1[t1.protocol == proto].set_index("features").folds.str.split(";").apply(lambda x: np.array(x, float))
    g = s[b] - s[a]; return pd.Series({"mean": g.mean(), "min": g.min(), "max": g.max(), "positive folds": int((g > 0).sum())})
pd.DataFrame({f"{a[:1]}→{b[:1]} {p}": fold_gain(a, b, p) for p in ("host", "spatial", "block")
              for a, b in [("A internal", "B A+all external"), ("C A+coordinates", "B A+all external")]}).T.round(3)

,mean,min,max,positive folds
A→B host,0.111,0.095,0.117,5.0
C→B host,0.000,-0.002,0.002,4.0
A→B spatial,0.110,0.081,0.152,5.0
C→B spatial,0.005,0.003,0.008,5.0
A→B block,0.117,0.049,0.176,5.0
C→B block,0.026,-0.006,0.077,4.0


## 2. Error in euros, with a cluster bootstrap interval
Clusters follow the CV grouping (hosts, 1 km cells, 3 km blocks); 500 resamples, model held fixed.

In [6]:
rng = np.random.default_rng(SEED); eur = []
for protocol, clus in (("host", "host_id"), ("spatial", "cell"), ("block", "block")):
    uh, inv = np.unique(df[clus].values, return_inverse=True); c = np.bincount(inv)
    pairs = [("A internal", k) for k in S if k.startswith("A+") or k == "B A+all external"] + \
            [("B A+all external", "C A+coordinates"), ("C A+coordinates", "D C+all external")] + [(k, "B A+all external") for k in S if k.startswith("B-")]
    for a, b in pairs:
        if (protocol, a) not in P or (protocol, b) not in P: continue
        ea, eb = np.abs(np.exp(P[(protocol, a)]) - df.price_eur.values), np.abs(np.exp(P[(protocol, b)]) - df.price_eur.values)
        s = np.bincount(inv, weights=ea - eb)
        bs = [(lambda i: s[i].sum() / c[i].sum())(rng.integers(0, len(uh), len(uh))) for _ in range(500)]
        eur.append({"protocol": protocol, "from": a, "to": b, "mae_from_eur": ea.mean(), "mae_to_eur": eb.mean(), "mae_reduction_eur": (ea - eb).mean(),
                    "ci95_low": np.percentile(bs, 2.5), "ci95_high": np.percentile(bs, 97.5), "clusters": len(uh)})
eur = pd.DataFrame(eur); eur.to_csv(OUT / "t1_price_mae_eur.csv", index=False)
eur.round(2)

,protocol,from,to,mae_from_eur,mae_to_eur,mae_reduction_eur,ci95_low,ci95_high,clusters
0,host,A internal,A+stations,87.96,81.27,6.69,6.09,7.28,22106
1,host,A internal,A+osm,87.96,77.19,10.77,9.90,11.50,22106
2,host,A internal,A+dvf,87.96,77.16,10.80,9.96,11.57,22106
3,host,A internal,A+iris_income,87.96,78.35,9.62,8.89,10.24,22106
4,host,A internal,A+crime,87.96,78.65,9.31,8.59,10.04,22106
5,host,A internal,B A+all external,87.96,76.54,11.43,10.58,12.14,22106
6,host,B A+all external,C A+coordinates,76.54,76.43,0.11,-0.13,0.39,22106
7,host,C A+coordinates,D C+all external,76.43,76.51,-0.08,-0.32,0.15,22106
8,host,B-stations,B A+all external,76.62,76.54,0.09,-0.02,0.18,22106
9,host,B-osm,B A+all external,76.57,76.54,0.03,-0.11,0.19,22106


**Reading.** `mae_reduction_eur > 0` means the `to` model has the lower error.
* A → B rows: the merging gain in EUR per night.
* B → C rows: negative means external data beat coordinates.
* B−source → B rows: positive means removing that source hurts, i.e. it carries information the other sources do not.

## 3. Two-stage price residual
Stage 1: listing attributes only (out-of-fold prediction). Stage 2: how much of what stage 1 cannot explain (the "location premium") is explained by location information alone.

In [7]:
r7 = []
for protocol in ("host", "spatial", "block"):
    res = y - P[(protocol, "A internal")]
    for label, cols in {"all external": EXT_ALL, **G, "coordinates + arrondissement": GEO}.items():
        p, _ = oof(df, cols, res, "reg", protocol)
        r7.append({"protocol": protocol, "stage 2 features": label, **score(res, p, "reg")})
r7 = pd.DataFrame(r7); r7.to_csv(OUT / "t7_residual.csv", index=False)
r7.pivot(index="stage 2 features", columns="protocol", values="R2")[["host", "spatial", "block"]].round(3)

protocol,host,spatial,block
stage 2 features,,,
all external,0.219,0.215,0.211
coordinates + arrondissement,0.215,0.208,0.127
crime,0.178,0.165,0.139
dvf,0.208,0.201,0.197
iris_income,0.189,0.157,0.121
osm,0.210,0.194,0.169
stations,0.149,0.102,0.088


In [8]:
pd.DataFrame({"id": df.id, "host_id": df.host_id, "price_eur": df.price_eur,
              **{f"{p}|{k}": v for (p, k), v in P.items() if k in CORE}}).to_csv(OUT / "t1_oof_predictions.csv.gz", index=False)